# Osoba A: BERT (HerBERT) + PEFT

Zakres (Część I, ~3,25 pkt): I.A, pkt 1, 2 i 4 z [instrukcji](../Zadanie_2_zmodyfikowane.md).

## Do zrobienia

1. **I.A**: zapoznać się z klasami `BertModel`, `BertForMaskedLM`, `BertForSequenceClassification`, `BertForTokenClassification` (krótka notatka w notebooku: co zwracają, jaka głowica, jakie wejścia).
2. **Pkt 1: strojenie z adapterem PEFT** (np. LoRA), model `allegro/herbert-base-cased`:
   - a. klasyfikacja tekstu (`BertForSequenceClassification`),
   - b. klasyfikacja tokenów (`BertForTokenClassification`),
   - c. ocena na zbiorze testowym.
3. **Pkt 2: wizualizacja przestrzeni wektorowej** zbioru testowego (tylko klasyfikacja tekstu), **przed** i **po** dostrojeniu. Reprezentacja tekstu = wektor tokenu `[CLS]`. Dwa interaktywne wykresy kolorowane etykietą i opis w markdownie, jak strojenie zmienia układ punktów.
4. **Pkt 4: augmentacja przez `[MASK]`** (`BertForMaskedLM`): maskujemy losowe słowa w tekstach treningowych i podstawiamy predykcje modelu. Potem trenujemy model na zbiorze powiększonym.

## Ustalone (nie zmieniamy bez uzgodnienia z B)

- Model: `allegro/herbert-base-cased`.
- Etykiety tekstu: `plus_m, minus_m, zero, amb` (`LABELS`, `LABEL2ID`). Tagi tokenów: BIO `wzmocnienie / oslabienie / negacja` (`TAGS`, `TAG2ID`).
- **Zbiór testowy jest wspólny z osobą B i nigdy nie jest augmentowany.**
  - tekst: `load_text('test')` (`all.text.test.txt`); zbiór treningowy: `load_text('train')`, walidacyjny: `load_text('dev')`,
  - tokeny: `split_tokens(load_tokens())` (seed 42).
- Augmentacja dotyczy tylko zbioru treningowego. Wynik zapisujemy w `../data/aug_bert.jsonl` (pola `text`, `label`, gdzie `label` to nazwa etykiety, a nie liczba). Finalny model trenujemy na `train + aug`.
- Adaptery i checkpointy zapisujemy w `../artifacts/` (nie w repo).

## Części wspólne (`../utils/`)

| Co | Skąd |
|---|---|
| wczytanie tekstów, etykiety | `utils.data`: `load_text`, `LABELS`, `LABEL2ID` |
| tagi BIO i podział train/test dla tokenów | `utils.tokens`: `load_tokens`, `split_tokens`, `TAGS`, `TAG2ID` |
| tagi słów → subtokeny (`-100`) | `utils.align`: `encode_tokens` |
| wykres przestrzeni wektorowej | `utils.plots`: `plot_embeddings` |

## Uwagi

- **Klasyfikacja tokenów ma mało danych**: anotacje z Zadania 1 to tylko 35 tekstów, więc wyniki będą niestabilne. Opisz to w wnioskach. Rozszerzenie (np. słownik z anotacji) ustalamy osobno z zespołem.
- Metryki: tekst → accuracy i macro-F1 (`sklearn`); tokeny → precision/recall/F1 per tag, z pominięciem pozycji `-100`.
- Wizualizacja: wektor `[CLS]` z ostatniej warstwy ukrytej (przed głowicą klasyfikacyjną). Dla modelu z adapterem PEFT weź go z modelu po `merge`/z wyłączonymi głowicami, tak jak dla bazowego.


In [7]:
import sys

sys.path.insert(0, "..")
from utils.align import encode_tokens
from utils.data import LABEL2ID, LABELS, load_text
from utils.plots import plot_embeddings
from utils.tokens import TAG2ID, TAGS, load_tokens, split_tokens

MODEL = "allegro/herbert-base-cased"
train, dev, test = load_text("train"), load_text("dev"), load_text("test")
tok_train, tok_test = split_tokens(load_tokens())
len(train), len(dev), len(test), len(tok_train), len(tok_test)

(6573, 823, 820, 28, 7)

## I.A: klasy BERT (notatka)

Wszystkie cztery klasy dzielą to samo ciało `model.bert` (`BertModel`). Różnią się tylko małą głowicą na wierzchu i funkcją straty. Wymiary podane są dla `herbert-base-cased`: 12 warstw, `hidden=768`, słownik 50 000 tokenów. `B` to batch, `T` to liczba subtokenów (z `[CLS]`, czyli `<s>`, i `[SEP]`, czyli `</s>`, oraz paddingiem).

**Wejścia (wspólne):** `input_ids` `(B, T)`, `attention_mask` `(B, T)` (1 = prawdziwy token, 0 = padding) i opcjonalnie `token_type_ids`. Klasy z głowicą przyjmują też `labels`. Jeśli je podamy, zwracają `loss`.

| Klasa | Głowica | `logits` / wyjście | `labels` → `loss` |
|---|---|---|---|
| `BertModel` | brak (tylko enkoder) + **pooler**: `tanh(Linear(768→768))` na wektorze `[CLS]` | `last_hidden_state` `(B, T, 768)`, `pooler_output` `(B, 768)` | brak |
| `BertForMaskedLM` | `BertOnlyMLMHead`: `Linear(768→768)` → GELU → LayerNorm → `decoder Linear(768→50000)` (wagi związane z embeddingami wejściowymi). Bez poolera. | `(B, T, 50000)`: rozkład po słowniku **dla każdej pozycji** | `(B, T)`, id prawdziwego tokenu na pozycjach `[MASK]`, `-100` w pozostałych. CrossEntropy po słowniku. |
| `BertForSequenceClassification` | `pooler_output` → dropout → `Linear(768→num_labels)` | `(B, 4)`: jeden wektor na **cały tekst** (u nas `plus_m, minus_m, zero, amb`) | `(B,)` z id klas. CrossEntropy (MSE, gdy `num_labels=1`; BCE dla multi-label). |
| `BertForTokenClassification` | `last_hidden_state` → dropout → `Linear(768→num_labels)`. Bez poolera. | `(B, T, 7)`: wektor klas **dla każdego subtokenu** (u nas 7 tagów BIO) | `(B, T)`, `-100` na `[CLS]`, `[SEP]`, paddingu i nie-pierwszych subtokenach słowa (`utils.align.encode_tokens`). CrossEntropy z `ignore_index=-100`. |

**Uwagi praktyczne:**
- `logits` to surowe wyniki, a nie prawdopodobieństwa. Klasa to `argmax`, a prawdopodobieństwa daje `softmax`.
- Przy `num_labels`, który różni się od checkpointu, `classifier` jest losowo inicjalizowany (ostrzeżenie „newly initialized”). Trzeba go trenować, a przy PEFT oznacza to dodanie go do `modules_to_save`.
- Do pkt 2 bierzemy surowy `[CLS]` = `last_hidden_state[:, 0]`, **a nie** `pooler_output` (ten jest dodatkowo przepuszczony przez Linear i tanh).
- W tokenizerze HerBERT `[CLS]`, `[SEP]` i `[MASK]` mają postać `<s>`, `</s>` i `<mask>`. W kodzie używamy `tok.cls_token`, `tok.mask_token` itd.

In [8]:
import torch
from transformers import (
    AutoTokenizer,
    BertForMaskedLM,
    BertForSequenceClassification,
    BertForTokenClassification,
    BertModel,
)

tok = AutoTokenizer.from_pretrained(MODEL)
batch = tok(
    ["Hotel był bardzo czysty.", "Nie polecam."], padding=True, return_tensors="pt"
)
B, T = batch["input_ids"].shape

with torch.no_grad():
    out = BertModel.from_pretrained(MODEL)(**batch)
    assert out.last_hidden_state.shape == (B, T, 768)
    assert out.pooler_output.shape == (B, 768)
    for cls, kw, shape in [
        (BertForMaskedLM, {}, (B, T, tok.vocab_size)),
        (BertForSequenceClassification, {"num_labels": len(LABELS)}, (B, len(LABELS))),
        (BertForTokenClassification, {"num_labels": len(TAGS)}, (B, T, len(TAGS))),
    ]:
        m = cls.from_pretrained(MODEL, **kw)
        assert m(**batch).logits.shape == shape
        print(
            f"{cls.__name__:32} logits {tuple(shape)}  głowica: {getattr(m, 'cls', None) or m.classifier}"
        )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.sso.sso_relationship.bias              | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.sso.sso_relationship.weight            | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: allegro/herbert-base-cased
Key                             | Status     |  | 
--------------------------------+------------+--+-
bert.pooler.dense.weight        | UNEXPECTED |  | 
cls.sso.sso_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias          | UNEXPECTED |  | 
cls.sso.sso_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM                  logits (2, 7, 50000)  głowica: BertOnlyMLMHead(
  (predictions): BertLMPredictionHead(
    (transform): BertPredictionHeadTransform(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (transform_act_fn): GELUActivation()
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    )
    (decoder): Linear(in_features=768, out_features=50000, bias=True)
  )
)


...

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

BertForSequenceClassification    logits (2, 4)  głowica: Linear(in_features=768, out_features=4, bias=True)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be

BertForTokenClassification       logits (2, 7, 7)  głowica: Linear(in_features=768, out_features=7, bias=True)


## 1a. Klasyfikacja tekstu + PEFT

HerBERT + LoRA (`query`, `value`, `alpha = 2r`) dla trzech rang `r ∈ {4, 8, 16}`. Wspólny trening: `utils/train.py` (`max_length=400`, 3 epoki, `lr=2e-4`). Najlepszy checkpoint każdego przebiegu wybieramy po **macro-F1 na dev**, a test służy tylko do końcowej oceny. Adaptery zapisujemy w `../artifacts/bert-lora-r{r}/best`.

**Baseline (`r=0`):** zamrożony HerBERT, trenujemy tylko `classifier` (na `pooler_output`). LoRA startuje z `B = 0`, więc `ΔW = B·A = 0`, a po zamrożeniu parametrów `lora_*` zostaje zerem. Ten sam trening co dla LoRA, a jedyna różnica to brak adapterów. Pokazuje, ile wiedzy o klasach jest w modelu **przed** strojeniem.

In [29]:
import gc
from pathlib import Path

import pandas as pd
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
)

from utils.train import make_dataset, plot_history
from utils.train import train as fit


ART = Path("../artifacts")


def free_gpu():
    """Po `del model, trainer`: oddaje cache alokatora CUDA, który inaczej zostaje zarezerwowany."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def make_model(
    r, auto=AutoModelForSequenceClassification, task=TaskType.SEQ_CLS, labels=LABELS
):
    model = auto.from_pretrained(
        MODEL,
        num_labels=len(labels),
        id2label=dict(enumerate(labels)),
        label2id={l: i for i, l in enumerate(labels)},
        # MPS: fused attention (sdpa) nie obsługuje dropoutu, gdy model jest w pełni zamrożony
        attn_implementation="eager" if r == 0 else "sdpa",
    )
    cfg = LoraConfig(
        task_type=task,
        r=max(r, 1),
        lora_alpha=2 * max(r, 1),
        lora_dropout=0.1,
        target_modules=["query", "value"],
    )
    model = get_peft_model(
        model, cfg
    )  # SEQ_CLS/TOKEN_CLS: `classifier` trafia do modules_to_save, więc też się trenuje
    if r == 0:  # baseline: B=0 i zamrożone, więc ΔW = 0; uczy się tylko `classifier`
        for name, p in model.named_parameters():
            if "lora_" in name:
                p.requires_grad = False
    return model


MAX_LEN = 400
tok = AutoTokenizer.from_pretrained(MODEL)
train_ds, dev_ds, test_ds = (make_dataset(d, tok, MAX_LEN) for d in (train, dev, test))


Map:   0%|          | 0/6573 [00:00<?, ? examples/s]

Map:   0%|          | 0/823 [00:00<?, ? examples/s]

Map:   0%|          | 0/820 [00:00<?, ? examples/s]

In [10]:
RANKS = [0, 4, 8, 16]  # 0 = baseline (zamrożony HerBERT)
EPOCHS = 5
results, histories, preds = [], {}, {}
for r in RANKS:
    name = "baseline (r=0)" if r == 0 else f"LoRA r={r}"
    model = make_model(r)
    model.print_trainable_parameters()
    trainer = fit(model, tok, train_ds, dev_ds, ART / f"bert-lora-r{r}", epochs=EPOCHS)
    trainer.save_model(ART / f"bert-lora-r{r}" / "best")
    out = trainer.predict(test_ds)
    histories[name] = trainer.state.log_history
    preds[r] = out.predictions.argmax(-1)
    results.append(
        {
            "model": name,
            "r": r,
            "trainable_params": sum(
                p.numel() for p in model.parameters() if p.requires_grad
            ),
            "train_time_s": next(
                log["train_runtime"]
                for log in trainer.state.log_history
                if "train_runtime" in log
            ),
            "dev_f1": trainer.state.best_metric,
            "test_accuracy": out.metrics["test_accuracy"],
            "test_f1": out.metrics["test_f1"],
        }
    )
    del model, trainer
    free_gpu()

results = pd.DataFrame(results)
results


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

trainable params: 3,076 || all params: 124,485,896 || trainable%: 0.0025


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 0}.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.203235,1.211593,0.442284,0.279197
2,1.148464,1.141702,0.563791,0.480060
3,1.120353,1.100276,0.558931,0.477276
4,1.075910,1.078578,0.586877,0.507437
5,1.101028,1.072643,0.588092,0.512581


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

trainable params: 592,900 || all params: 125,038,856 || trainable%: 0.4742


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.377131,0.365820,0.845687,0.841045
2,0.287873,0.340178,0.866343,0.848886
3,0.289029,0.342125,0.869988,0.851774
4,0.168688,0.327047,0.880923,0.872567
5,0.237311,0.339582,0.880923,0.872249


,model,r,trainable_params,train_time_s,dev_f1,test_accuracy,test_f1
0,baseline (r=0),0,3076,86.6177,0.512581,0.613415,0.533046
1,LoRA r=4,4,150532,109.4270,0.876059,0.886585,0.870244
2,LoRA r=8,8,297988,108.9326,0.877043,0.880488,0.867788
3,LoRA r=16,16,592900,108.6980,0.872567,0.887805,0.871723


In [11]:
plot_history(histories, "Krzywe uczenia: baseline vs HerBERT + LoRA")

### 1aa. Porównanie na zbiorze testowym: baseline vs LoRA (każde `r`)

In [12]:
import plotly.express as px

px.bar(
    results.melt(
        id_vars="model",
        value_vars=["test_accuracy", "test_f1"],
        var_name="metryka",
        value_name="wynik",
    ),
    x="model",
    y="wynik",
    color="metryka",
    barmode="group",
    text_auto=".3f",
    title="Zbiór testowy: baseline (przed strojeniem) vs HerBERT + LoRA",
)

**Wnioski 1a** _(do uzupełnienia po pełnym treningu)_: ile daje strojenie względem baseline'u, wpływ `r` na jakość i czas, przebieg krzywych (czy jest przeuczenie).

Wprowadzenie jakiekolwiek r do eksperymentu, duzo wprowadzilo. Sam BERT (baseline) nie ma az tak dobrego wyczucia sentymentu. Tzn gdy dotrenowywaliśmy sama glowice klasyfikatora, to 1. wolno sie uczyla 2. nie wiadomo czy osiagnela by wyniki porownywalne z LORa.

Wybór poziomu r: nie mial duzego znaczenia. Czasowo wyszly bardzo podobnie

## 1b. Klasyfikacja tokenów + PEFT

HerBERT + LoRA na tagach BIO (`wzmocnienie`, `oslabienie`, `negacja`; 7 tagów). Ten sam schemat co w 1a: `r=0` to baseline (zamrożony HerBERT, uczy się tylko `classifier`), a `r ∈ {4, 8, 16}` to LoRA. Dane z `load_tokens()` to 35 tekstów; `split_tokens` daje 28 train / 7 test, a z train wydzielamy kilka tekstów na dev (wybór checkpointu).

**Metryka:** `f1` = macro-F1 po 6 tagach **bez `O`**, liczony na pierwszych subtokenach słów (pozycje `-100` pomijamy). `O` to ~96% tokenów, więc accuracy i F1 z `O` byłyby zawyżone samym przewidywaniem `O`. Na końcu raport per tag.

In [18]:
from sklearn.model_selection import train_test_split
from transformers import DataCollatorForTokenClassification

from utils.train import make_token_dataset, token_metrics

RANKS_TOK = [0, 4, 8, 16]  # 0 = baseline (zamrożony HerBERT)
EPOCHS_TOK, BATCH_TOK, LR_TOK = 200, 8, 1e-3
MAX_LEN_TOK, N_DEV_TOK = 512, 4  # najdłuższy tekst: 411 subtokenów

tok_tr, tok_dev = train_test_split(tok_train, test_size=N_DEV_TOK, random_state=42)
tok_train_ds, tok_dev_ds, tok_test_ds = (
    make_token_dataset(d, tok, MAX_LEN_TOK) for d in (tok_tr, tok_dev, tok_test)
)
len(tok_train_ds), len(tok_dev_ds), len(tok_test_ds)

(24, 4, 7)

In [19]:
results_tok, histories_tok, outs_tok = [], {}, {}
for r in RANKS_TOK:
    name = "baseline (r=0)" if r == 0 else f"LoRA r={r}"
    model = make_model(r, AutoModelForTokenClassification, TaskType.TOKEN_CLS, TAGS)
    model.print_trainable_parameters()
    trainer = fit(
        model,
        tok,
        tok_train_ds,
        tok_dev_ds,
        ART / f"bert-tok-lora-r{r}",
        epochs=EPOCHS_TOK,
        lr=LR_TOK,
        batch_size=BATCH_TOK,
        collator=DataCollatorForTokenClassification(tok),
        compute_metrics=token_metrics,
    )
    trainer.save_model(ART / f"bert-tok-lora-r{r}" / "best")
    out = trainer.predict(tok_test_ds)
    histories_tok[name], outs_tok[name] = trainer.state.log_history, out
    results_tok.append(
        {
            "model": name,
            "r": r,
            "trainable_params": sum(
                p.numel() for p in model.parameters() if p.requires_grad
            ),
            "train_time_s": next(
                log["train_runtime"]
                for log in trainer.state.log_history
                if "train_runtime" in log
            ),
            "dev_f1": trainer.state.best_metric,
            "test_accuracy": out.metrics["test_accuracy"],
            "test_f1": out.metrics["test_f1"],
        }
    )
    del model, trainer
    free_gpu()

results_tok = pd.DataFrame(results_tok)
results_tok


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be

trainable params: 5,383 || all params: 123,899,918 || trainable%: 0.0043


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 0}.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,1.514116,0.603340,0.004535
2,No log,1.060339,0.954071,0.000000
3,No log,0.713555,0.972860,0.000000
4,1.204691,0.470210,0.972860,0.000000
5,1.204691,0.320084,0.972860,0.000000
6,1.204691,0.234469,0.972860,0.000000
7,1.204691,0.189569,0.972860,0.000000
8,0.293419,0.168185,0.972860,0.000000
9,0.293419,0.159182,0.972860,0.000000
10,0.293419,0.154188,0.972860,0.000000


...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be

trainable params: 595,207 || all params: 124,452,878 || trainable%: 0.4783


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,1.005362,0.960334,0.000000
2,No log,0.177663,0.972860,0.000000
3,No log,0.176201,0.972860,0.000000
4,0.735914,0.173712,0.972860,0.000000
5,0.735914,0.166140,0.972860,0.000000
6,0.735914,0.165572,0.972860,0.000000
7,0.735914,0.162659,0.972860,0.000000
8,0.171122,0.157719,0.972860,0.000000
9,0.171122,0.153143,0.972860,0.000000
10,0.171122,0.146939,0.972860,0.000000


,model,r,trainable_params,train_time_s,dev_f1,test_accuracy,test_f1
0,baseline (r=0),0,5383,80.2707,0.152778,0.970896,0.134310
1,LoRA r=4,4,152839,85.6501,0.240741,0.975553,0.237671
2,LoRA r=8,8,300295,106.7109,0.269841,0.983702,0.387632
3,LoRA r=16,16,595207,86.1285,0.244444,0.975553,0.317180


In [20]:
plot_history(
    histories_tok, "1b. Krzywe uczenia: baseline vs HerBERT + LoRA (dev macro-F1 bez O)"
)

### 1bb. Porównanie na zbiorze testowym i raport per tag

In [21]:
from sklearn.metrics import classification_report

px.bar(
    results_tok.melt(
        id_vars="model",
        value_vars=["test_accuracy", "test_f1"],
        var_name="metryka",
        value_name="wynik",
    ),
    x="model",
    y="wynik",
    color="metryka",
    barmode="group",
    text_auto=".3f",
    title="1b. Zbiór testowy: accuracy (z O) i macro-F1 (bez O)",
).show()

for name, out in outs_tok.items():
    mask = out.label_ids != -100
    print(f"=== {name}")
    print(
        classification_report(
            out.label_ids[mask],
            out.predictions.argmax(-1)[mask],
            labels=range(len(TAGS)),
            target_names=TAGS,
            digits=3,
            zero_division=0,
        )
    )

=== baseline (r=0)
               precision    recall  f1-score   support

            O      0.972     0.999     0.985       829
B-wzmocnienie      1.000     0.444     0.615         9
I-wzmocnienie      0.000     0.000     0.000         1
 B-oslabienie      0.000     0.000     0.000         2
 I-oslabienie      0.000     0.000     0.000         0
    B-negacja      0.667     0.111     0.190        18
    I-negacja      0.000     0.000     0.000         0

     accuracy                          0.971       859
    macro avg      0.377     0.222     0.256       859
 weighted avg      0.962     0.971     0.961       859

=== LoRA r=4
               precision    recall  f1-score   support

            O      0.983     0.992     0.987       829
B-wzmocnienie      1.000     0.889     0.941         9
I-wzmocnienie      0.000     0.000     0.000         1
 B-oslabienie      0.000     0.000     0.000         2
 I-oslabienie      0.000     0.000     0.000         0
    B-negacja      0.533     

**Wnioski 1b** _(do uzupełnienia po treningu)_: czy model wychodzi poza przewidywanie samego `O`, które tagi rozpoznaje (`B-negacja`?), wpływ `r` i liczby kroków treningu (24 teksty), niestabilność wyników przy 7 tekstach testowych.

Trzeba było mocno zwiekszyc liczbe epok i lr=1e-3. WWynika to z tego, ze mielismy tylko 24 teksty treningowe.

Na accuracy nie ma co patrzec, bo zbiór jest mocno niezbalansowany. 

r=8 wyszo najlepiej - ale wynik f1 =0.39 jest całkiem słaby.

## 2. Wizualizacja przestrzeni wektorowej (`[CLS]`): przed i po strojeniu

Reprezentacja tekstu to surowy wektor `[CLS]` z ostatniej warstwy (`last_hidden_state[:, 0]`, przed `classifier`), liczony dla zbioru testowego.

- **przed:** surowy HerBERT (`BertModel`),
- **po:** HerBERT z adapterem LoRA z 1a (`../artifacts/bert-lora-r{r}/best`, `r = 4, 8, 16`), po `merge_and_unload()`, czyli `W + B·A` wpisane w wagi.

Baseline `r=0` ma identyczny `[CLS]` co „przed” (`ΔW = 0`), więc służy tylko jako test poprawności ładowania. Wykresy: t-SNE do 2D. **Silhouette** (cosinus) liczony w pełnej przestrzeni 768D, a nie na t-SNE, bo t-SNE zniekształca odległości. Wartość bliżej 1 oznacza zwarte i rozdzielone klasy, ~0 oznacza klasy wymieszane. Osie t-SNE są liczone osobno dla każdego wykresu, więc porównujemy strukturę, a nie współrzędne.

In [22]:
import numpy as np
from peft import PeftModel
from sklearn.metrics import silhouette_score

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)


@torch.no_grad()
def cls_embeddings(bert, texts, batch_size=32):
    """Surowy wektor [CLS] z ostatniej warstwy: last_hidden_state[:, 0] -> (len(texts), 768)."""
    bert.eval().to(DEVICE)
    out = []
    for i in range(0, len(texts), batch_size):
        batch = tok(
            texts[i : i + batch_size],
            truncation=True,
            max_length=MAX_LEN,
            padding=True,
            return_tensors="pt",
        )
        out.append(bert(**batch.to(DEVICE)).last_hidden_state[:, 0].cpu())
    return torch.cat(out).numpy()


def load_lora_bert(r):
    """HerBERT + adapter z 1a, scalony w zwykły BertModel."""
    base = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=len(LABELS)
    )
    return (
        PeftModel.from_pretrained(base, ART / f"bert-lora-r{r}" / "best")
        .merge_and_unload()
        .bert
    )


texts = list(test["text"])
embs = {"przed: HerBERT": cls_embeddings(BertModel.from_pretrained(MODEL), texts)}
# test poprawności: r=0 nie zmienia [CLS] (B = 0, uczy się tylko classifier)
assert np.allclose(
    cls_embeddings(load_lora_bert(0), texts[:32]),
    embs["przed: HerBERT"][:32],
    atol=1e-4,
)
for r in [r for r in RANKS if r > 0]:
    embs[f"po: LoRA r={r}"] = cls_embeddings(load_lora_bert(r), texts)

silhouette = pd.DataFrame(
    [
        {
            "model": name,
            "silhouette (cosine, 768D)": silhouette_score(
                e, test["label"], metric="cosine"
            ),
        }
        for name, e in embs.items()
    ]
)
silhouette

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.sso.sso_relationship.bias              | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.sso.sso_relationship.weight            | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

,model,"silhouette (cosine, 768D)"
0,przed: HerBERT,-0.008246
1,po: LoRA r=4,0.471564
2,po: LoRA r=8,0.496493
3,po: LoRA r=16,0.510519


In [23]:
for name, e in embs.items():
    plot_embeddings(
        e,
        test["label"],
        texts,
        title=f"{name}: t-SNE wektorów [CLS] (test)",
        method="tsne",
    ).show()

**Wnioski 2** _(do uzupełnienia po treningu 1a)_: jak wyglądają klasy przed strojeniem i po nim, gdzie leży `amb` i `zero`, jak zmienia się silhouette wraz z `r`.

Po strojemu mamy silhouete na poziomie mniej wiecej 0.5. Najbardziej odseparowana klasa jest zawsze klasa zero. Dobrze widoczne jest przejscie pomiedzy minus, amb, az do plus

## 4. Augmentacja przez `[MASK]` i trening na zbiorze powiększonym

Augmentacja tylko zbioru treningowego, przez `BertForMaskedLM` (bazowy HerBERT). Dla każdego tekstu robimy **2 kopie**. W każdej kopii zmieniamy **7% słów** (co najmniej 1), wybranych losowo spośród słów z literami w części widocznej dla modelu (`MAX_LEN` subtokenów). Słowa zmieniamy **po kolei**: maskujemy jedno słowo (`<mask>`), model proponuje słowo, podstawiamy je i dopiero wtedy maskujemy następne, więc każde podstawienie widzi poprzednie. Podstawienie losujemy z **top-4** (proporcjonalnie do prawdopodobieństwa) spośród tokenów będących całym słowem z literami (`...</w>`), z pominięciem oryginalnego słowa.

Wynik: `../data/aug_bert.jsonl` (`text`, `label` + `changes` do kontroli jakości). Jeśli plik istnieje, wczytujemy go zamiast generować. Potem trenujemy LoRA `r=8` na `train + aug` i porównujemy z `r=8` z 1a na tym samym teście.

In [31]:
from tqdm.auto import tqdm

from utils.data import detokenize, load_aug

AUG_FRAC, AUG_COPIES, AUG_TOPK, AUG_BATCH = 0.07, 2, 4, 8
AUG_PATH = Path("../data/aug_bert.jsonl")

mlm = BertForMaskedLM.from_pretrained(MODEL).eval().to(DEVICE)
vocab = tok.convert_ids_to_tokens(list(range(mlm.config.vocab_size)))
# dozwolone podstawienia: całe słowo (`</w>` na końcu) zawierające literę
allowed = torch.tensor([t.endswith("</w>") and any(ch.isalpha() for ch in t.removesuffix("</w>")) for t in vocab], device=DEVICE)


@torch.no_grad()
def augment(raws, seed):
    """Jedna kopia każdego tekstu (kolumna `raw`, słowa rozdzielone spacjami) -> (teksty, lista zmian [(stare, nowe)])."""
    rng, gen = np.random.default_rng(seed), torch.Generator().manual_seed(seed)
    words = [r.split() for r in raws]
    plans = []  # dla każdego tekstu: indeksy słów do zmiany, w kolejności podstawiania
    for w in tqdm(words, desc=f"plan (seed={seed})", leave=False):
        visible = sorted({i for i in tok(w, is_split_into_words=True, truncation=True, max_length=MAX_LEN).word_ids() if i is not None})
        cand = [i for i in visible if any(ch.isalpha() for ch in w[i])]
        k = min(len(cand), max(1, round(AUG_FRAC * len(cand))))
        plans.append(list(rng.choice(cand, size=k, replace=False)))
    changes = [[] for _ in raws]
    bar = tqdm(total=sum(map(len, plans)), desc=f"podstawienia (seed={seed})", unit="słowo")
    for step in range(max(map(len, plans))):
        todo = sorted((j for j, p in enumerate(plans) if step < len(p)), key=lambda j: len(words[j]))
        for b in range(0, len(todo), AUG_BATCH):
            ids = todo[b : b + AUG_BATCH]
            pos = [plans[j][step] for j in ids]
            masked = [" ".join(words[j][:i] + [tok.mask_token] + words[j][i + 1 :]) for j, i in zip(ids, pos)]
            enc = tok(masked, truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEVICE)
            logits = mlm(**enc).logits[enc["input_ids"] == tok.mask_token_id]  # (len(ids), vocab): jedna maska na tekst
            logits = logits.masked_fill(~allowed, -torch.inf)
            for row, (j, i) in enumerate(zip(ids, pos)):
                orig = tok.convert_tokens_to_ids(words[j][i] + "</w>")
                if orig != tok.unk_token_id:
                    logits[row, orig] = -torch.inf
            top = logits.topk(AUG_TOPK)
            pick = torch.multinomial(top.values.softmax(-1).cpu(), 1, generator=gen).squeeze(1)
            for row, (j, i) in enumerate(zip(ids, pos)):
                new = vocab[top.indices[row, pick[row]]].removesuffix("</w>")
                changes[j].append((words[j][i], new))
                words[j][i] = new
            bar.update(len(ids))
    bar.close()
    return [detokenize(" ".join(w)) for w in words], changes


if AUG_PATH.exists():
    aug = pd.read_json(AUG_PATH, lines=True)
else:
    parts = []
    for c in range(AUG_COPIES):
        texts_aug, changes = augment(list(train["raw"]), seed=42 + c)
        parts.append(pd.DataFrame({"text": texts_aug, "label": train["label"].values, "changes": changes}))
    aug = pd.concat(parts, ignore_index=True)
    aug.to_json(AUG_PATH, orient="records", lines=True, force_ascii=False)
len(aug), aug["changes"].str.len().describe()


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: allegro/herbert-base-cased
Key                             | Status     |  | 
--------------------------------+------------+--+-
bert.pooler.dense.weight        | UNEXPECTED |  | 
cls.sso.sso_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias          | UNEXPECTED |  | 
cls.sso.sso_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


plan (seed=42):   0%|          | 0/6573 [00:00<?, ?it/s]

podstawienia (seed=42):   0%|          | 0/49728 [00:00<?, ?słowo/s]

plan (seed=43):   0%|          | 0/6573 [00:00<?, ?it/s]

podstawienia (seed=43):   0%|          | 0/49728 [00:00<?, ?słowo/s]

(13146,
 count    13146.000000
 mean         7.565495
 std          3.886817
 min          1.000000
 25%          5.000000
 50%          7.000000
 75%          9.000000
 max         22.000000
 Name: changes, dtype: float64)

### 4a. Kontrola jakości augmentacji

In [32]:
for _, row in aug.sample(10, random_state=0).iterrows():
    print(f"[{row['label']}] " + ", ".join(f"{a} → {b}" for a, b in row["changes"]))
    print("   ", row["text"][:300], "\n")

[minus_m] na → od, Zdjęcia → Jak, w → na, dla → na, zewnątrz → kuchni
    Zarezerwował em pobyt w tym obiekcie przez Booking, na miejscu okazało się iż oferowanego mi pokoju nie ma, zaproponowano mi pokój bez okna od kuchni, brudny nie sprzątany nie wiem jak długo, z rozpadającym się łóżkiem na dzieci, ręcznikami jak ścierki do podłogi. Tregedia i oszustwo. Booking przeni 

[zero] o → m, jaskry → choroby, zamykającym → wysokim, pacjentów → ludzi
    Kierowani przez singapurskich badaczy prof. Tina Aunga i prof. CC. Khora naukowcy przebadali ponad 10, 5 tys. ludzi z jaskrą pierwotną z wysokim kątem przesączania. Wśród Azjatów jest to najczęstszy rodzaj choroby - cierpi na nią ponad połowa osób z jaskrą. W Polsce natomiast choroba występuje rzadz 

[plus_m] D3 → D, bardzo → silne, USG → badania, razem → zawsze, dowiedziała → okazało, na → w, dodatkowy → taki, pantofle → koszulki, oddać → pobrać
    Bardzo polecam Panią Doktor. Po wielu latach ciągłego uczucia zmęczenia wyprowadziła mnie 

### 4b. LoRA `r=8` na `train + aug` vs `r=8` z 1a (ten sam test)

In [33]:
R_AUG = 8
train_aug_ds = make_dataset(pd.concat([train, load_aug("bert")]), tok, MAX_LEN)

model = make_model(R_AUG)
model.print_trainable_parameters()
trainer = fit(model, tok, train_aug_ds, dev_ds, ART / f"bert-lora-r{R_AUG}-aug", epochs=EPOCHS)
trainer.save_model(ART / f"bert-lora-r{R_AUG}-aug" / "best")
out = trainer.predict(test_ds)
del model, trainer
free_gpu()

aug_vs = pd.concat([
    results.loc[results["r"] == R_AUG, ["model", "test_accuracy", "test_f1"]],
    pd.DataFrame([{"model": f"LoRA r={R_AUG} + aug", "test_accuracy": out.metrics["test_accuracy"], "test_f1": out.metrics["test_f1"]}]),
])
px.bar(
    aug_vs.melt(id_vars="model", value_vars=["test_accuracy", "test_f1"], var_name="metryka", value_name="wynik"),
    x="model", y="wynik", color="metryka", barmode="group", text_auto=".3f",
    title=f"Zbiór testowy: LoRA r={R_AUG} bez augmentacji vs z augmentacją [MASK]",
)


Map:   0%|          | 0/19719 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: allegro/herbert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.sso.sso_relationship.bias              | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.sso.sso_relationship.weight            | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

trainable params: 297,988 || all params: 124,743,944 || trainable%: 0.2389


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 0}.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.314588,0.343056,0.872418,0.862460
2,0.243839,0.320977,0.874848,0.863301
3,0.211935,0.346579,0.874848,0.870019
4,0.186101,0.372738,0.884569,0.874935
5,0.150124,0.385206,0.885784,0.876335


**Wnioski 4** _(do uzupełnienia po treningu)_: jak wyglądają podstawienia (czy zmieniają sentyment?), czy augmentacja pomogła względem `r=8` z 1a, koszt (3× więcej danych treningowych).

Augmentacja troszkę pomogła.